In [ ]:
# %% [CELL 1] Configuration  <- the only cell you edit (SMOKE)
# =============================================================================
# STAGE 2 v4 — STEP 1: v3 + "which words differ" features, more negatives, larger model
# Amazon ML Challenge 2026 — Business Entity Resolution
# =============================================================================
# NEW vs v3 (from the v3 diagnostics: blank-address and sibling-word errors)
#   * extra / missing words (no close counterpart in the other name) and the
#     legal-form change, each target-encoded with a learned match rate:
#     training rows use rates from the OTHER half of training entities,
#     held-out / test rows use rates from all training entities (no leakage)
#   * more negatives in training (0.3 -> 0.5), larger model (255 leaves)
# Held-out split and all other steps identical to v3 (v3: F0.5 0.9644).
# OUTPUT  /kaggle/working/stage2_models_v4/ (+ te_stats.json, the learned rates)
#
# v3 CHANGES KEPT FROM v2
#   * candidates = original blocking (L1-L4) + Phase B (L5-L7) + Phase C (L8, L2t)
#   * pre-filter features from all 11 layer scores (incl. L8 name / address parts)
#   * non-Latin records use their IndicTrans2 translation in every name feature
#   * phonetic skeleton with the order bug fixed
#   * candidates kept per entity chosen per country from held-out retention
#   * streams queries in chunks: ~190M train pairs never sit in memory at once
# Held-out split = hash buckets 0-499, the SAME entities as v2 (v2: F0.5 0.9432),
# so the two results are directly comparable.
#
# INPUTS  competition dataset, ckpt_train zips, Phase B train output (ckpt_b_train),
#         Phase C train output (ckpt_c_train), translations_train.parquet
# RUN     SMOKE = True first, then SMOKE = False (interactive or Save & Run All)
# SETTINGS Accelerator None (CPU) is enough; GPU also fine. Internet ON.
#          Persistence Files. Running in the SAME notebook as Step 1 v2 reuses its
#          prepared texts (prep_train/) and saves ~7 min.
# =============================================================================

SMOKE = True
SMOKE_QUERIES = 30_000        # smoke: first queries per country

DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

Q_CHUNK = 50_000              # queries per streamed chunk
PF_QUERIES = 150_000          # first queries per country used to train the pre-filter
PF_NEG_RATE = 0.15
KEEP_OPTIONS = (20, 30, 40, 50, 60, 80)
RETENTION_TARGET = 0.995      # keep >= 99.5% of blocked true pairs (else the largest option)
MATCHER_NEG_RATE = 0.5        # share of non-matching kept pairs used to train the matcher (v3: 0.3)
PRED_CHUNK = 5_000_000
FEAT_CHUNK = 2_000_000
MAX_ROUNDS = 3000
VAL_BUCKETS, FIT_END, ES_END = 500, 3000, 800   # held out 0-499 | train 500-2999 (ES 500-799)
V2_HELDOUT = 0.9432           # v2 on the same held-out entities, for comparison
V3_HELDOUT = 0.9644           # v3 on the same held-out entities
SEED = 42
MATCHER_PARAMS = {"objective": "binary", "num_leaves": 255, "learning_rate": 0.06,
                  "min_data_in_leaf": 100, "feature_fraction": 0.8, "bagging_fraction": 0.8,
                  "bagging_freq": 1, "lambda_l2": 1.0, "seed": SEED, "verbose": -1}

In [ ]:
# %% [CELL 2] Imports and helpers
# ---------------------------------------------------------------------------
# Logging, timing, zip-aware input finder.
# ---------------------------------------------------------------------------
import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import lightgbm as lgb
from tqdm.auto import tqdm

try:
    from rapidfuzz.process import cpdist
except ImportError:
    print("installing/upgrading rapidfuzz (needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz"], check=True)
    try:
        from rapidfuzz.process import cpdist
    except ImportError:
        raise SystemExit("rapidfuzz was upgraded: restart the session, then rerun this cell.")
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

DATA_ROOT, WORK, INPUT_ROOT = Path(DATA_ROOT), Path(WORK), Path(INPUT_ROOT)
T_START = time.time()
TIMES = {}                     # stage -> [seconds, scalable?]


def log(msg):
    print(f"  [{(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)


def mem(tag=""):
    rss = psutil.Process().memory_info().rss / 1e9
    print(f"    [MEM] {tag:<28} RAM={rss:5.1f} GB  free={psutil.virtual_memory().available/1e9:5.1f} GB",
          flush=True)


def bar(it=None, **kw):
    kw.setdefault("mininterval", 10)
    return tqdm(it, **kw)


def timed(stage, t0, scalable=True):
    """Accumulate wall time per stage. scalable: True = grows with the number of
    pairs, False = fixed cost, None = done once and reused by the full run."""
    TIMES.setdefault(stage, [0.0, scalable])[0] += time.time() - t0


def hash_bucket(eid):
    """Deterministic 0-9999 bucket; identical to the EDA / Notebook 2 split."""
    return int(hashlib.md5(eid.encode()).hexdigest(), 16) % 10000


def locate(marker):
    """All files named `marker` under INPUT_ROOT, plus those inside any .zip under
    INPUT_ROOT. Each zip holding the marker is unpacked once into
    WORK/_unzipped/<zip name>/ (Kaggle may keep uploaded zips packed)."""
    root_ok = INPUT_ROOT.exists()
    hits = sorted(INPUT_ROOT.rglob(marker)) if root_ok else []
    for z in (sorted(INPUT_ROOT.rglob("*.zip")) if root_ok else []):
        dest = WORK / "_unzipped" / z.stem
        done = dest / ".unpacked"
        if not done.exists():
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("/" + marker) or n == marker for n in zf.namelist()):
                    continue
                log(f"unpacking {z.name} -> {dest}")
                zf.extractall(dest)
            done.touch()
        hits += sorted(dest.rglob(marker))
    return hits


def project(full_factor, label):
    """Print measured stage times and their projection to the full run."""
    print(f"\n  RUNTIME PROJECTION ({label})")
    print(f"    {'stage':<30}{'smoke (s)':>11}{'full (min)':>12}")
    total = 0.0
    for stage, (sec, scalable) in TIMES.items():
        # scalable None = already done and reused by the full run
        full = 0.0 if scalable is None else sec * (full_factor if scalable else 1.0)
        total += full
        print(f"    {stage:<30}{sec:>11.1f}{full/60:>12.1f}")
    print(f"    {'TOTAL':<30}{'':>11}{total/60:>12.1f}")
    return total

In [ ]:
# %% [CELL 3] Text cleaning (shared with every stage)
# ---------------------------------------------------------------------------
# Identical cleaning everywhere.
# ---------------------------------------------------------------------------
COLS = ["entity_id", "business_name", "business_address", "country"]


def load_src(path):
    """Read a source TSV. sep='\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS, header=0, keep_default_na=False)


# ---- cleaning: copied verbatim from Notebook 2 so both stages agree ----------
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")

ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]


def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


# ---- extra text columns for Stage 2 -------------------------------------------
# Street-type words and French articles are skipped when forming the address key,
# so "15 Rue de la Hoxie" -> "15_hoxie" (not "15_rue"), matching "351 Hoxie Avenue"
# style keys. This matters most for France, which training never sees.
SKIP_AFTER_NUMBER = {
    "road", "street", "drive", "avenue", "boulevard", "lane", "court", "circle",
    "way", "highway", "rue", "impasse", "allee", "chemin", "place", "route", "quai",
    "cours", "square", "de", "du", "des", "la", "le", "les", "l", "d",
}


def addr_key_string(caddr):
    """House number + the first following token that isn't a street word/article.
    Returned as a space-joined, sorted string so rapidfuzz can compare two of them."""
    toks = caddr.split()
    keys = set()
    for i, a in enumerate(toks):
        if a.isdigit():
            a2 = a.lstrip("0")
            j = i + 1
            while j < len(toks) and toks[j] in SKIP_AFTER_NUMBER:
                j += 1
            if a2 and j < len(toks):
                b = toks[j].lstrip("0") if toks[j].isdigit() else toks[j]
                if b:
                    keys.add(a2 + "_" + b)
    return " ".join(sorted(keys))


def prepare_texts(df):
    """All per-record text columns Stage 2 needs, computed once."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["digits"] = out["caddr"].str.findall(r"\b\d+\b").str.join(" ")
    out["nospace"] = out["cname"].str.replace(" ", "", regex=False)
    out["akeys"] = [addr_key_string(a) for a in out["caddr"]]
    out["nonlatin"] = np.array([any(ord(ch) > 0x024F for ch in s)
                                for s in df["business_name"]], dtype=np.int8)
    return out

In [ ]:
# %% [CELL 4] String features (shared)
# ---------------------------------------------------------------------------
# rapidfuzz similarities + flags.
# ---------------------------------------------------------------------------
# ---- feature definitions (shared with Step 2) --------------------------------
def group_features(q, S):
    """Scores relative to each S1's own candidate list (q must be sorted).
    Scale-free features like rank and gap-to-best transfer across countries."""
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    sizes = np.diff(np.r_[starts, len(q)])
    cols = []
    for j in range(S.shape[1]):
        s = S[:, j].astype(np.float32)
        gmax = np.repeat(np.maximum.reduceat(s, starts), sizes)
        order = np.lexsort((-s, q))
        rank = np.empty(len(q), np.float32)
        rank[order] = np.arange(len(q)) - np.repeat(starts, sizes)
        cols += [s, gmax - s, rank]
    cols += [(S[:, :3] > 0).sum(1) + (S[:, 3] > 0), np.repeat(sizes, sizes)]
    return np.column_stack(cols).astype(np.float32)


SCORERS = [("cname", "ratio", fuzz.ratio), ("cname", "token_set", fuzz.token_set_ratio),
           ("cname", "token_sort", fuzz.token_sort_ratio), ("cname", "partial", fuzz.partial_ratio),
           ("cname", "jaro_winkler", JaroWinkler.normalized_similarity),
           ("nospace", "ratio", fuzz.ratio), ("caddr", "token_set", fuzz.token_set_ratio),
           ("caddr", "token_sort", fuzz.token_sort_ratio), ("caddr", "ratio", fuzz.ratio),
           ("digits", "token_set", fuzz.token_set_ratio), ("akeys", "token_set", fuzz.token_set_ratio)]
FLAG_FEATURES = ["a_blank", "q_blank", "p_blank", "nonlatin", "len_diff"]
PRE_FEATURES = [f"{s}_{k}" for s in ("l1", "l2", "l3", "ak") for k in ("score", "gap", "rank")] \
    + ["n_layers", "n_cands", "is_s3"]
RICH_FEATURES = PRE_FEATURES + ["pf", "pf_gap", "pf_rank"] \
    + [f"{col}_{nm}" for col, nm, _ in SCORERS] + FLAG_FEATURES


def string_features(qt, pt, q, p):
    """String similarities + flags for pairs (q[i], p[i]); qt/pt are the prepared
    text tables aligned to qids/pids. Chunked to bound memory."""
    Q = {c: qt[c].to_numpy() for c in ("cname", "caddr", "digits", "nospace", "akeys")}
    P = {c: pt[c].to_numpy() for c in ("cname", "caddr", "digits", "nospace", "akeys")}
    qlen, plen = qt["cname"].str.len().to_numpy(), pt["cname"].str.len().to_numpy()
    q_nl, p_nl = qt["nonlatin"].to_numpy(), pt["nonlatin"].to_numpy()
    out = np.empty((len(q), len(SCORERS) + len(FLAG_FEATURES)), dtype=np.float32)
    for i in bar(range(0, len(q), FEAT_CHUNK), desc="string features", unit="chunk"):
        qs, ps = q[i:i + FEAT_CHUNK], p[i:i + FEAT_CHUNK]
        n = len(qs)
        for k, (col, _, sc) in enumerate(SCORERS):
            out[i:i + n, k] = cpdist(Q[col][qs].tolist(), P[col][ps].tolist(),
                                     scorer=sc, workers=-1)
        j = len(SCORERS)
        out[i:i + n, j] = (Q["caddr"][qs] == "") | (P["caddr"][ps] == "")
        out[i:i + n, j + 1] = Q["cname"][qs] == ""
        out[i:i + n, j + 2] = P["cname"][ps] == ""
        out[i:i + n, j + 3] = q_nl[qs] | p_nl[ps]
        out[i:i + n, j + 4] = np.abs(qlen[qs] - plen[ps])
    return out

In [ ]:
# %% [CELL 5] v2 features (shared)
# ---------------------------------------------------------------------------
# Installs indic_transliteration if missing.
# ---------------------------------------------------------------------------
# ---- v2: transliteration, legal form, phonetic skeleton (shared) -------------
import re as _re
import unicodedata as _ud
try:
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
except ImportError:
    print("installing indic_transliteration (MIT licence; needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "indic_transliteration"], check=True)
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit

_BLOCK_SCRIPT = {0x0900: _sanscript.DEVANAGARI, 0x0980: _sanscript.BENGALI,
                 0x0A00: _sanscript.GURMUKHI, 0x0A80: _sanscript.GUJARATI,
                 0x0B00: _sanscript.ORIYA, 0x0B80: _sanscript.TAMIL,
                 0x0C00: _sanscript.TELUGU, 0x0C80: _sanscript.KANNADA,
                 0x0D00: _sanscript.MALAYALAM}
_LEGAL_TOKENS = {"pvt", "private", "pr", "pra", "ltd", "limited", "li", "llp", "pllc", "llc",
                 "lp", "inc", "incorporated", "corp", "corporation", "co", "company", "sas",
                 "sasu", "sarl", "eurl", "snc", "sa", "public"}
_LEGAL_PREFIX = ("prai", "pirai", "limi", "elel")    # transliterated private / limited / LLP
_FORM_KEYS = (("LLP", {"llp"}), ("PLLC", {"pllc"}), ("LLC", {"llc"}), ("LP", {"lp"}),
              ("INC", {"inc", "incorporated"}), ("CORP", {"corp", "corporation"}),
              ("SASU", {"sasu"}), ("SAS", {"sas"}), ("SARL", {"sarl"}), ("EURL", {"eurl"}),
              ("SNC", {"snc"}), ("SA", {"sa"}))
_SKEL_FOLD = (("ph", "f"), ("th", "t"), ("sh", "s"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
              ("dh", "d"), ("ch", "c"), ("ck", "k"), ("c", "k"), ("q", "k"), ("x", "ks"),
              ("z", "s"), ("w", "v"), ("j", "g"), ("y", ""), ("v", "b"))   # Bengali renders B as v


def latin_tokens(name):
    """Lowercase Latin tokens of a name. Indic-script words are transliterated
    (ITRANS), accents removed, dots dropped (L.L.C -> llc), other punctuation
    splits words (Housing-Network -> housing network)."""
    s = str(name)
    s = _re.sub(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", " ", s, flags=_re.I)
    s = s.replace(".", "")
    fast = all(ord(ch) <= 0x024F for ch in s)
    if fast:
        s = _re.sub(r"[^\w\s]", " ", s)
    else:
        # Indic vowel signs / virama are not \w, so only replace real punctuation
        # and symbols (Unicode categories P* and S*).
        s = "".join(" " if _ud.category(ch)[0] in "PS" else ch for ch in s)
    out = []
    for w in s.split():
        indic = False
        if not fast:
            blk = next((b for b in _BLOCK_SCRIPT if any(b <= ord(c) < b + 0x80 for c in w)), None)
            if blk is not None:
                indic = True
                try:
                    w = _translit(w, _BLOCK_SCRIPT[blk], _sanscript.ITRANS)
                except Exception:
                    pass
                if blk == 0x0B80:                     # Tamil: library leaves some letters
                    w = w.replace("\u0BA9", "n").replace("\u0BB1", "r")
        w = _ud.normalize("NFKD", w).encode("ascii", "ignore").decode().lower()
        if indic and blk == 0x0B80:
            # Tamil has no aspirated/voiced contrast; the library writes ச க ப ட/த
            # as jh gh bh dh — fold them to s k p t.
            for a, b in (("jh", "s"), ("gh", "k"), ("bh", "p"), ("dh", "t")):
                w = w.replace(a, b)
        w = _re.sub(r"[^a-z0-9]", "", w)
        if indic:
            w = _re.sub(r"([bcdfghjklmnpqrstvwxyz])a$", r"\1", w)    # inherent final 'a'
        if w:
            out.append(w)
    return out


def legal_form(toks):
    """Canonical legal form: PVT (private limited), PLC, LTD, LLP, LLC, INC, ...
    Spelling variants map to one code (Pvt Ltd = Private Limited = प्रा. लि.)."""
    s = set(toks)
    has_pvt = bool(s & {"pvt", "private", "pr", "pra"}) or any(t.startswith(("prai", "pirai")) for t in toks)
    has_ltd = bool(s & {"ltd", "limited", "li"}) or any(t.startswith("limi") for t in toks)
    if has_pvt:
        return "PVT"
    if "public" in s and has_ltd:
        return "PLC"
    if any(t.startswith("elel") for t in toks):
        return "LLP"
    for code, keys in _FORM_KEYS:
        if s & keys:
            return code
    if has_ltd:
        return "LTD"
    if s & {"co", "company"}:
        return "CO"
    return ""


def skeleton(toks):
    """Phonetic consonant skeleton of the non-legal tokens: folds spelling variants,
    drops vowels, collapses repeats. 'teknolojis' and 'technologies' -> 'tknlgs'."""
    words = [t for t in toks if t not in _LEGAL_TOKENS and not t.startswith(_LEGAL_PREFIX)]
    s = " ".join(words)
    for a, b in _SKEL_FOLD:
        s = s.replace(a, b)
    s = _re.sub(r"[aeiou]", "", s)
    s = _re.sub(r"(.)\1+", r"\1", s).replace("m", "n")
    return _re.sub(r"\s+", " ", s).strip()


def prepare_extra(names):
    """Per-record legal form and phonetic skeleton."""
    forms, skels = [], []
    for nm in names:
        toks = latin_tokens(nm)
        forms.append(legal_form(toks))
        skels.append(skeleton(toks))
    return pd.DataFrame({"form": forms, "skel": skels})


EXTRA_FEATURES = ["form_same", "form_diff", "form_one_missing", "name_exact",
                  "skel_ratio", "skel_tset"]
RICH_V2 = RICH_FEATURES + EXTRA_FEATURES          # v1 features first, then v2 additions


def string_features_v2(qt, pt, q, p):
    """v1 string features + legal form relation, exact name, skeleton similarity.
    qt/pt must include the v2 columns 'form' and 'skel'."""
    base = string_features(qt, pt, q, p)
    fq, fp = qt["form"].to_numpy()[q], pt["form"].to_numpy()[p]
    cq, cp = qt["cname"].to_numpy()[q], pt["cname"].to_numpy()[p]
    sq, sp = qt["skel"].to_numpy()[q].tolist(), pt["skel"].to_numpy()[p].tolist()
    both = (fq != "") & (fp != "")
    extra = np.column_stack([
        both & (fq == fp),
        both & (fq != fp),
        (fq == "") != (fp == ""),
        (cq != "") & (cq == cp),
        cpdist(sq, sp, scorer=fuzz.ratio, workers=-1),
        cpdist(sq, sp, scorer=fuzz.token_set_ratio, workers=-1),
    ]).astype(np.float32)
    return np.column_stack([base, extra]).astype(np.float32)


def ensure_extra(prep_dir, ids_path, side, raw_names_by_id):
    """Cache <side>_x.parquet (form, skel) aligned to ids; compute if missing."""
    path = prep_dir / f"{side}_x.parquet"
    if path.exists():
        return
    ids = np.load(ids_path, allow_pickle=True)
    names = raw_names_by_id.reindex(ids)
    assert names.notna().all(), f"{path}: ids missing from the source files"
    t = time.time()
    x = prepare_extra(names.tolist())
    x.insert(0, "entity_id", ids)
    x.to_parquet(path, index=False)
    log(f"{path.parent.name}/{path.name}: {len(ids):,} rows in {time.time()-t:.0f}s")


def load_prepared(prep_dir, side):
    """v1 prepared text table + v2 extra columns, row-aligned."""
    t = pd.read_parquet(prep_dir / f"{side}.parquet")
    x = pd.read_parquet(prep_dir / f"{side}_x.parquet")
    assert (t["entity_id"].values == x["entity_id"].values).all(), "extra columns misaligned"
    t["form"], t["skel"] = x["form"].values, x["skel"].values
    return t

In [ ]:
# %% [CELL 6] v3 shared code
# ---------------------------------------------------------------------------
# All blocking layers, merge, translated names, fixed skeleton.
# ---------------------------------------------------------------------------
# ---- v3: all blocking layers, translated names, fixed skeleton (Step 1/2 v3) ----
import os
from concurrent.futures import ProcessPoolExecutor as _PPE
import multiprocessing as _mp3

LAYERS = ["l1", "l2", "l3", "ak", "l5", "l6", "l7", "l8", "l8n", "l8a", "l2t"]
LAYER_SRC = {"old": ["l1", "l2", "l3", "ak"], "b": ["l5", "l6", "l7"], "c": ["l8", "l8n", "l8a", "l2t"]}
PRE_V3 = [f"{s}_{k}" for s in LAYERS for k in ("score", "gap", "rank")] + ["n_layers", "n_cands", "is_s3"]
RICH_V3 = (PRE_V3 + ["pf", "pf_gap", "pf_rank"] + [f"{col}_{nm}" for col, nm, _ in SCORERS]
           + FLAG_FEATURES + EXTRA_FEATURES + ["p_translated"])


def group_features_v3(q, S):
    """Per-query relative features for every layer score (q sorted)."""
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    sizes = np.diff(np.r_[starts, len(q)])
    cols = []
    for j in range(S.shape[1]):
        s = S[:, j].astype(np.float32)
        gmax = np.repeat(np.maximum.reduceat(s, starts), sizes)
        order = np.lexsort((-s, q))
        rank = np.empty(len(q), np.float32)
        rank[order] = np.arange(len(q)) - np.repeat(starts, sizes)
        cols += [s, gmax - s, rank]
    cols += [(S > 0).sum(1), np.repeat(sizes, sizes)]
    return np.column_stack(cols).astype(np.float32)


def load_sources(paths):
    """paths: {"old": pairs.npz, "b": pairs_b.npz, "c": pairs_c.npz} -> arrays in memory."""
    D = {}
    for src, path in paths.items():
        z = np.load(path)
        D[src] = {k: z[k] for k in z.files}
    return D


def merge_chunk(D, q0, q1, npool):
    """Union of the candidate sources for queries q0..q1-1, sorted by (q, p);
    one column per layer score (0 when a source did not propose the pair)."""
    keys, parts = [], []
    for src, cols in LAYER_SRC.items():
        if src not in D:
            continue
        d = D[src]
        lo, hi = np.searchsorted(d["q"], [q0, q1])
        keys.append(d["q"][lo:hi].astype(np.int64) * npool + d["p"][lo:hi])
        parts.append((src, lo, hi))
    allk = np.concatenate(keys)
    uk, inv = np.unique(allk, return_inverse=True)
    out = {"q": (uk // npool).astype(np.int32), "p": (uk % npool).astype(np.int32),
           "S": np.zeros((len(uk), len(LAYERS)), dtype=np.float32)}
    has_label = all("label" in D[s] for s, _, _ in parts)
    if has_label:
        out["y"] = np.zeros(len(uk), dtype=np.int8)
    off = 0
    for src, lo, hi in parts:
        n = hi - lo
        idx = inv[off:off + n]                 # unique within a source: plain assignment
        for c in LAYER_SRC[src]:
            if c in D[src]:
                out["S"][idx, LAYERS.index(c)] = D[src][c][lo:hi]
        if has_label:
            out["y"][idx] = np.maximum(out["y"][idx], D[src]["label"][lo:hi])
        off += n
    return out


def skeleton3(toks):
    """Phonetic skeleton with the order fixed (m->n BEFORE collapsing repeats)."""
    words = [t for t in toks if t not in _LEGAL_TOKENS and not t.startswith(_LEGAL_PREFIX)]
    s = " ".join(words)
    for a, b in _SKEL_FOLD:
        s = s.replace(a, b)
    s = _re.sub(r"[aeiou]", "", s).replace("m", "n")
    s = _re.sub(r"(.)\1+", r"\1", s)
    return _re.sub(r"\s+", " ", s).strip()


def _x3_task(args):
    names, trans = args
    forms, skels = [], []
    for n, t in zip(names, trans):
        tr = latin_tokens(n)
        forms.append(legal_form(tr))           # form read from the original (translations may shorten it)
        skels.append(skeleton3(latin_tokens(t) if t else tr))
    return forms, skels


def ensure_x3(prep_dir, side, ids, names_by_id, trans):
    """Cache <side>_x3.parquet: legal form, fixed skeleton, and the cleaned name
    with translations substituted for non-Latin records."""
    path = prep_dir / f"{side}_x3.parquet"
    if path.exists():
        return
    t = time.time()
    names = names_by_id.reindex(ids)
    assert names.notna().all(), f"{path}: ids missing from the source files"
    names = names.tolist()
    tr = [trans.get(x, "") for x in ids]
    tasks = [(names[i:i + 100_000], tr[i:i + 100_000]) for i in range(0, len(names), 100_000)]
    forms, skels = [], []
    with _PPE(max_workers=os.cpu_count() or 2, mp_context=_mp3.get_context("fork")) as ex:
        for f_, s_ in ex.map(_x3_task, tasks):
            forms += f_; skels += s_
    base = pd.read_parquet(prep_dir / f"{side}.parquet", columns=["entity_id", "cname"])
    assert (base["entity_id"].values == ids).all(), "prepared texts misaligned"
    cname_t = base["cname"].to_numpy().copy()
    ti = np.array([i for i, x in enumerate(tr) if x], dtype=np.int64)
    if len(ti):
        cname_t[ti] = clean_name(pd.Series([tr[i] for i in ti])).to_numpy()
    x = pd.DataFrame({"entity_id": ids, "form": forms, "skel3": skels, "cname_t": cname_t})
    x["nospace_t"] = x["cname_t"].str.replace(" ", "", regex=False)
    x["translated"] = np.array([1 if v else 0 for v in tr], dtype=np.int8)
    x.to_parquet(path, index=False)
    log(f"{prep_dir.name}/{path.name}: {len(ids):,} rows ({int(x['translated'].sum()):,} translated) "
        f"in {time.time()-t:.0f}s")


def load_prepared_v3(prep_dir, side):
    """Prepared texts with v3 columns: translated cleaned names, fixed skeleton."""
    t = pd.read_parquet(prep_dir / f"{side}.parquet")
    x = pd.read_parquet(prep_dir / f"{side}_x3.parquet")
    assert (t["entity_id"].values == x["entity_id"].values).all(), "v3 columns misaligned"
    t["cname"], t["nospace"] = x["cname_t"].values, x["nospace_t"].values
    t["form"], t["skel"], t["translated"] = x["form"].values, x["skel3"].values, x["translated"].values
    return t


def string_features_v3(qt, pt, q, p):
    """v2 string features computed on translated names + fixed skeleton,
    plus a flag saying the candidate's name came from a translation."""
    base = string_features_v2(qt, pt, q, p)
    return np.column_stack([base, pt["translated"].to_numpy()[p].astype(np.float32)]).astype(np.float32)

In [ ]:
# %% [CELL 7] v4 shared code: word-difference features (shared with Step 2 v4)
# ---------------------------------------------------------------------------
# Extra / missing words and form change, target-encoded.
# ---------------------------------------------------------------------------
# ---- v4: "which words differ" features, target-encoded (Step 1/2 v4) ------------
TE_FEATURES = ["ex_cnt", "ex_min", "ex_mean", "ex_max", "mi_cnt", "mi_min", "mi_mean", "mi_max", "form_tr"]
RICH_V4 = RICH_V3 + TE_FEATURES
TE_A = 20          # smoothing: a word seen n times gets (pos + TE_A*prior) / (n + TE_A)
TE_MIN_N = 5       # words seen fewer times use the prior (kept out of the saved table)
TE_FUZZ = 80       # a word has a counterpart if string similarity >= 80 (words of 4+ letters)
_DT = {}


def te_prepare(qt, pt):
    """Word sets of every query / pool record (cleaned names; translations for non-Latin)."""
    ws = lambda t: [frozenset(w for w in s.split() if len(w) >= 2) for s in t["cname"].tolist()]
    _DT.clear()
    _DT.update(QW=ws(qt), PW=ws(pt), QF=qt["form"].to_numpy(), PF=pt["form"].to_numpy())


def _has_counterpart(w, others):
    if w in others:
        return True
    if len(w) < 4:
        return False
    return any(len(s) >= 4 and fuzz.ratio(w, s) >= TE_FUZZ for s in others)


def _diff_task(bounds):
    lo, hi = bounds
    q, p, QW, PW = _DT["q"][lo:hi], _DT["p"][lo:hi], _DT["QW"], _DT["PW"]
    ex, mi = [], []
    for a, b in zip(q, p):
        A, B = QW[a], PW[b]
        ex.append(" ".join(sorted(w for w in B - A if not _has_counterpart(w, A))))
        mi.append(" ".join(sorted(w for w in A - B if not _has_counterpart(w, B))))
    return ex, mi


def diff_tokens(q, p, chunk=200_000):
    """Per pair: extra words (candidate only), missing words (S1 only), form change.
    Runs in parallel over CPU cores (forked workers read _DT without copying)."""
    _DT["q"], _DT["p"] = q, p
    tasks = [(i, min(len(q), i + chunk)) for i in range(0, len(q), chunk)]
    ex, mi = [], []
    with _PPE(max_workers=os.cpu_count() or 2, mp_context=_mp3.get_context("fork")) as pool:
        for e_, m_ in pool.map(_diff_task, tasks):
            ex += e_; mi += m_
    fr = _DT["QF"][q].astype(object) + ">" + _DT["PF"][p].astype(object)
    return ex, mi, fr


def te_stats(ex, mi, fr, y):
    """Match-rate table for extra words, missing words and form changes."""
    st = {}
    for kind, vals, split in (("ex", ex, True), ("mi", mi, True), ("fr", fr, False)):
        n, pos, rows_n, rows_pos = defaultdict(int), defaultdict(float), 0, 0.0
        for v, yy in zip(vals, y):
            toks = v.split() if split else [v]
            if not toks:
                continue
            rows_n += 1; rows_pos += yy
            for t in toks:
                n[t] += 1; pos[t] += yy
        st[kind] = {t: [n[t], pos[t]] for t in n if n[t] >= TE_MIN_N}
        st["prior_" + kind] = rows_pos / max(rows_n, 1)
    return st


def te_features(ex, mi, fr, st):
    """9 features per pair from a rate table (NaN where there is no word)."""
    out = np.full((len(ex), len(TE_FEATURES)), np.nan, dtype=np.float32)
    rate = {k: {t: (v[1] + TE_A * st["prior_" + k]) / (v[0] + TE_A) for t, v in st[k].items()}
            for k in ("ex", "mi", "fr")}
    for i, (e, m, f) in enumerate(zip(ex, mi, fr)):
        for j, (kind, v) in enumerate((("ex", e), ("mi", m))):
            toks = v.split()
            out[i, 4 * j] = len(toks)
            if toks:
                r = [rate[kind].get(t, st["prior_" + kind]) for t in toks]
                out[i, 4 * j + 1] = min(r); out[i, 4 * j + 2] = sum(r) / len(r); out[i, 4 * j + 3] = max(r)
        out[i, 8] = rate["fr"].get(f, st["prior_fr"])
    return out

In [ ]:
# %% [CELL 8] Inputs
def by_country(marker, parent):
    return {f.parent.name: f.parent for f in locate(marker) if f.parent.parent.name == parent}

CDIR = {c: d for c, d in by_country("qids.npy", "ckpt_train").items() if (d / "pairs.npz").exists()}
BDIR = by_country("pairs_b.npz", "ckpt_b_train")
CCDIR = by_country("pairs_c.npz", "ckpt_c_train")
assert CDIR, "ckpt_train not found: attach the ckpt_train zips"
assert BDIR, "ckpt_b_train not found: attach the Phase B train output"
assert CCDIR, "ckpt_c_train not found: attach the Phase C train output"
tr_files = sorted(INPUT_ROOT.rglob("translations_train.parquet")) + sorted(WORK.glob("translations_train.parquet"))
assert tr_files, "translations_train.parquet not found: attach the translation (train) output"
_t = pd.read_parquet(tr_files[0], columns=["entity_id", "translation"])
TRANS = dict(zip(_t["entity_id"], _t["translation"]))
del _t
COUNTRIES = sorted(set(CDIR) & set(BDIR) & set(CCDIR))
assert COUNTRIES, "no country has all three candidate sources"
PREP = WORK / "prep_train"
OUT = WORK / ("stage2_models_v4_smoke" if SMOKE else "stage2_models_v4")
PREP.mkdir(exist_ok=True)
print("=" * 70)
print(f"STEP 1 v4  {'SMOKE RUN' if SMOKE else 'FULL RUN'}   countries={COUNTRIES}   "
      f"translations={len(TRANS):,}")
print("=" * 70)

In [ ]:
# %% [CELL 9] Prepared texts (v1 tables reused if present) + v3 columns (cached)
t0 = time.time()
need = [(c, s) for c in COUNTRIES for s in ("q", "p") if not (PREP / c / f"{s}.parquet").exists()]
need3 = [(c, s) for c in COUNTRIES for s in ("q", "p") if not (PREP / c / f"{s}_x3.parquet").exists()]
if need or need3:
    recs = pd.concat([load_src(DATA_ROOT / "train" / f"train_source{i}.tsv") for i in (1, 2, 3)],
                     ignore_index=True)
    log(f"loaded {len(recs):,} train records")
    if need:
        row_of = pd.Index(recs["entity_id"])
        for c, side in need:
            (PREP / c).mkdir(parents=True, exist_ok=True)
            ids = np.load(CDIR[c] / f"{side}ids.npy", allow_pickle=True)
            rows = row_of.get_indexer(ids)
            assert (rows >= 0).all(), f"{c}/{side}: ids missing from the train files"
            prepare_texts(recs.iloc[rows]).to_parquet(PREP / c / f"{side}.parquet", index=False)
            log(f"[{c}] {side}: {len(ids):,} rows prepared")
        del row_of
    names = recs.set_index("entity_id")["business_name"]
    del recs
    for c, side in need3:
        ensure_x3(PREP / c, side, np.load(CDIR[c] / f"{side}ids.npy", allow_pickle=True), names, TRANS)
    del names
    gc.collect()
else:
    log("prepared texts (v1 + v3 columns) found — skipping")
timed("prepared texts (reused by full run)", t0, scalable=None)

In [ ]:
# %% [CELL 10] Pass A — train the pre-filter on the first PF_QUERIES queries
def is_val(b): return b < VAL_BUCKETS
def is_fit(b): return (b >= VAL_BUCKETS) & (b < FIT_END)
def is_es(b):  return (b >= VAL_BUCKETS) & (b < ES_END)

t0 = time.time()
rng = np.random.default_rng(SEED)
META, Xs, ys = {}, [], []
for c in COUNTRIES:
    qids = np.load(CDIR[c] / "qids.npy", allow_pickle=True)
    pids = np.load(CDIR[c] / "pids.npy", allow_pickle=True)
    nq = min(SMOKE_QUERIES, len(qids)) if SMOKE else len(qids)
    META[c] = dict(qids=qids[:nq], nq=nq, npool=len(pids), full_nq=len(qids),
                   qb=np.array([hash_bucket(x) for x in qids[:nq]], dtype=np.int16),
                   is_s3=np.fromiter((x.startswith("S3-") for x in pids), dtype=np.int8, count=len(pids)))
    D = load_sources({"old": CDIR[c] / "pairs.npz", "b": BDIR[c] / "pairs_b.npz", "c": CCDIR[c] / "pairs_c.npz"})
    for q0 in range(0, min(nq, PF_QUERIES), Q_CHUNK):
        q1 = min(nq, PF_QUERIES, q0 + Q_CHUNK)
        m = merge_chunk(D, q0, q1, META[c]["npool"])
        sel = is_fit(META[c]["qb"][m["q"]]) & ((m["y"] == 1) | (rng.random(len(m["y"])) < PF_NEG_RATE))
        X = np.column_stack([group_features_v3(m["q"], m["S"]), META[c]["is_s3"][m["p"]]]).astype(np.float32)
        Xs.append(X[sel]); ys.append(m["y"][sel])
        del m, X
    del D
    gc.collect()
Xs, ys = np.vstack(Xs), np.concatenate(ys)
prefilter = lgb.train({"objective": "binary", "num_leaves": 63, "learning_rate": 0.1,
                       "min_data_in_leaf": 100, "feature_fraction": 0.9, "seed": SEED, "verbose": -1},
                      lgb.Dataset(Xs, ys, feature_name=PRE_V3), 250)
log(f"pre-filter trained on {len(ys):,} rows ({ys.mean()*100:.1f}% positive)")
del Xs, ys
timed("pass A: pre-filter", t0)
mem("after pass A")

In [ ]:
# %% [CELL 11] Pass B — stream all queries: merge, pre-filter, keep top-60
t0 = time.time()
N_MAX = max(KEEP_OPTIONS)
KEPT, VALPOS = {}, {}
for c in COUNTRIES:
    mc = META[c]
    D = load_sources({"old": CDIR[c] / "pairs.npz", "b": BDIR[c] / "pairs_b.npz", "c": CCDIR[c] / "pairs_c.npz"})
    parts = {k: [] for k in ("q", "p", "y", "X", "pf", "gap", "rank")}
    pos_ranks, pos_all = [], np.zeros(mc["nq"], dtype=np.int64)
    n_pairs = 0
    for q0 in bar(range(0, mc["nq"], Q_CHUNK), desc=f"pass B {c}", unit="chunk"):
        q1 = min(mc["nq"], q0 + Q_CHUNK)
        m = merge_chunk(D, q0, q1, mc["npool"])
        n_pairs += len(m["q"])
        X = np.column_stack([group_features_v3(m["q"], m["S"]), mc["is_s3"][m["p"]]]).astype(np.float32)
        pf = np.concatenate([prefilter.predict(X[i:i + PRED_CHUNK])
                             for i in range(0, len(X), PRED_CHUNK)]).astype(np.float32)
        order = np.lexsort((-pf, m["q"]))
        qo = m["q"][order]
        st = np.flatnonzero(np.r_[True, qo[1:] != qo[:-1]])
        rank = np.arange(len(qo)) - np.repeat(st, np.diff(np.r_[st, len(qo)]))
        np.add.at(pos_all, m["q"][m["y"] == 1], 1)
        vpos = (m["y"][order] == 1) & is_val(mc["qb"][qo])
        pos_ranks.append(rank[vpos].astype(np.int32))
        sel = rank < N_MAX
        idx = order[sel]
        q_k, pf_k = m["q"][idx], pf[idx]
        s2 = np.flatnonzero(np.r_[True, q_k[1:] != q_k[:-1]])
        parts["q"].append(q_k); parts["p"].append(m["p"][idx]); parts["y"].append(m["y"][idx])
        parts["X"].append(X[idx].astype(np.float16)); parts["pf"].append(pf_k)
        parts["gap"].append(np.repeat(pf_k[s2], np.diff(np.r_[s2, len(q_k)])) - pf_k)
        parts["rank"].append(rank[sel].astype(np.int16))
        del m, X, pf, order
        gc.collect()
    KEPT[c] = {k: np.concatenate(v) for k, v in parts.items()}
    VALPOS[c] = (np.concatenate(pos_ranks), pos_all)
    mc["n_pairs"] = n_pairs
    log(f"[{c}] {mc['nq']:,} queries, {n_pairs:,} merged candidate pairs "
        f"({n_pairs/max(mc['nq'],1):.0f}/entity) -> kept top-{N_MAX}: {len(KEPT[c]['q']):,}")
    del D, parts
    gc.collect()
timed("pass B: stream + pre-filter", t0)
mem("after pass B")

In [ ]:
# %% [CELL 12] Pass C — keep-count per country, then string features for the rows used
print("\n  RETENTION of blocked true pairs on held-out entities (per country):")
N_KEEP = {}
for c in COUNTRIES:
    ranks, pos_all = VALPOS[c]
    tot = int(pos_all[is_val(META[c]["qb"])].sum())
    line, chosen = [], None
    for N in KEEP_OPTIONS:
        r = (ranks < N).sum() / max(tot, 1)
        line.append(f"top-{N} {r*100:6.2f}%")
        if chosen is None and r >= RETENTION_TARGET:
            chosen = N
    N_KEEP[c] = chosen or N_MAX
    print(f"    {c:<7} " + "  ".join(line) + f"   -> keep {N_KEEP[c]}")

t0 = time.time()
rng = np.random.default_rng(SEED + 1)
R = {}
for c in COUNTRIES:
    k = KEPT[c]
    keep = k["rank"] < N_KEEP[c]
    b = META[c]["qb"][k["q"]]
    use = keep & (is_val(b) | ((is_fit(b)) & ((k["y"] == 1) | (rng.random(len(k["y"])) < MATCHER_NEG_RATE))))
    ix = np.flatnonzero(use)
    qt, pt = load_prepared_v3(PREP / c, "q"), load_prepared_v3(PREP / c, "p")
    SF = string_features_v3(qt, pt, k["q"][ix], k["p"][ix])
    te_prepare(qt, pt)
    ex_, mi_, fr_ = diff_tokens(k["q"][ix], k["p"][ix])
    R[c] = dict(q=k["q"][ix], p=k["p"][ix], y=k["y"][ix], b=b[ix], ex=ex_, mi=mi_, fr=fr_,
                tef=np.array([hash_bucket(s + "#te") % 2 for s in META[c]["qids"][k["q"][ix]]], dtype=np.int8),
                X=np.column_stack([k["X"][ix].astype(np.float32), k["pf"][ix], k["gap"][ix],
                                   k["rank"][ix].astype(np.float32), SF]).astype(np.float32),
                kept_pos_val=np.bincount(k["q"][keep & (k["y"] == 1) & is_val(b)], minlength=META[c]["nq"]))
    log(f"[{c}] features for {len(ix):,} rows (all held-out + sampled training rows)")
    del qt, pt, SF
    KEPT[c] = None
    gc.collect()
timed("pass C: string features", t0)

# target-encoded word features: fit rows use the OTHER half's rates, held-out rows all fit rates
t0 = time.time()
def _gather(sel):
    ex, mi, fr, yy = [], [], [], []
    for c in COUNTRIES:
        r = R[c]
        for i in np.flatnonzero(sel(r)):
            ex.append(r["ex"][i]); mi.append(r["mi"][i]); fr.append(r["fr"][i]); yy.append(r["y"][i])
    return ex, mi, fr, yy

TE = {k: te_stats(*_gather(lambda r, k=k: is_fit(r["b"]) & (r["tef"] == k))) for k in (0, 1)}
TE["all"] = te_stats(*_gather(lambda r: is_fit(r["b"])))
for c in COUNTRIES:
    r = R[c]
    F = np.full((len(r["y"]), len(TE_FEATURES)), np.nan, dtype=np.float32)
    for key, sel in ((1, is_fit(r["b"]) & (r["tef"] == 0)), (0, is_fit(r["b"]) & (r["tef"] == 1)),
                     ("all", ~is_fit(r["b"]))):
        ix = np.flatnonzero(sel)
        if len(ix):
            F[ix] = te_features([r["ex"][i] for i in ix], [r["mi"][i] for i in ix], r["fr"][ix], TE[key])
    r["X"] = np.column_stack([r["X"], F]).astype(np.float32)
    del r["ex"], r["mi"]
assert R[COUNTRIES[0]]["X"].shape[1] == len(RICH_V4), "feature count mismatch"
log(f"word-difference rates: {len(TE['all']['ex']):,} extra words, {len(TE['all']['mi']):,} missing words, "
    f"{len(TE['all']['fr']):,} form changes (seen >= {TE_MIN_N} times)")
timed("pass C: word-difference features", t0)
mem("after pass C")

In [ ]:
# %% [CELL 13] Pass D — matcher
t0 = time.time()
def stack(sel):
    return (np.vstack([R[c]["X"][sel(R[c]["b"])] for c in COUNTRIES]),
            np.concatenate([R[c]["y"][sel(R[c]["b"])] for c in COUNTRIES]))

Xtr, ytr = stack(lambda b: is_fit(b) & ~is_es(b))
Xes, yes = stack(is_es)
log(f"matcher train rows {len(ytr):,} ({ytr.mean()*100:.1f}% positive), early-stop rows {len(yes):,}")
matcher = lgb.train(MATCHER_PARAMS, lgb.Dataset(Xtr, ytr, feature_name=RICH_V4), MAX_ROUNDS,
                    valid_sets=[lgb.Dataset(Xes, yes, feature_name=RICH_V4)],
                    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(250)])
log(f"matcher best iteration {matcher.best_iteration}")
del Xtr, ytr, Xes, yes
gc.collect()
timed("pass D: matcher training", t0)

In [ ]:
# %% [CELL 14] Held-out evaluation (same entities as v2)
t0 = time.time()
gt = pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                 names=["s1", "m"], header=0, keep_default_na=False)
truth_n = dict(zip(gt["s1"], gt["m"].map(lambda s: sum(1 for x in s.split(",") if x))))
del gt
E = dict(ent=[], prob=[], y=[], own=[])
tcs, blks, kps, ctys = [], [], [], []
off = 0
for i, c in enumerate(COUNTRIES):
    r, mc = R[c], META[c]
    vq = np.flatnonzero(is_val(mc["qb"]))
    ent_of = np.full(mc["nq"], -1, dtype=np.int64)
    ent_of[vq] = np.arange(off, off + len(vq))
    m = is_val(r["b"])
    prob = matcher.predict(r["X"][m], num_iteration=matcher.best_iteration).astype(np.float32)
    best = pd.Series(prob).groupby(r["p"][m]).transform("max").to_numpy()
    E["ent"].append(ent_of[r["q"][m]]); E["prob"].append(prob); E["y"].append(r["y"][m])
    E["own"].append(prob >= best)                       # one-owner among held-out entities
    tcs.append(np.array([truth_n[s] for s in mc["qids"][vq]], dtype=np.int64))
    blks.append(VALPOS[c][1][vq]); kps.append(r["kept_pos_val"][vq])
    ctys.append(np.full(len(vq), i))
    off += len(vq)
E = {k: np.concatenate(v) for k, v in E.items()}
tc, blk, kp, cty = map(np.concatenate, (tcs, blks, kps, ctys))
n_ent, pair_cty = len(tc), cty[E["ent"]]
GRID = np.round(np.arange(0.02, 0.99, 0.01), 2)


def f05(pc, tp, tc):
    P = np.where(pc > 0, tp / np.maximum(pc, 1), 0.0)
    R_ = np.where(tc > 0, tp / np.maximum(tc, 1), 0.0)
    f = np.where(P + R_ > 0, 1.25 * P * R_ / np.maximum(0.25 * P + R_, 1e-12), 0.0)
    return np.where((tc == 0) & (pc == 0), 1.0, f)


def per_entity(hit):
    return (np.bincount(E["ent"][hit], minlength=n_ent),
            np.bincount(E["ent"][hit], weights=E["y"][hit], minlength=n_ent).astype(np.int64))


def tune(allowed, per_country):
    if not per_country:
        t_ = float(max(GRID, key=lambda t: f05(*per_entity(allowed & (E["prob"] >= t)), tc).mean()))
        th = {c: t_ for c in COUNTRIES}
    else:
        th = {c: float(max(GRID, key=lambda t: f05(*per_entity(allowed & (pair_cty == i) & (E["prob"] >= t)),
                                                   tc)[cty == i].mean()))
              for i, c in enumerate(COUNTRIES)}
    thr = np.array([th[c] for c in COUNTRIES])[pair_cty]
    pc, tp = per_entity(allowed & (E["prob"] >= thr))
    return th, f05(pc, tp, tc), pc, tp


allp = np.ones(len(E["prob"]), bool)
results = {"v4, one threshold": tune(allp, False),
           "v4, per-country thresholds": tune(allp, True),
           "v4, per-country + one-owner (pessimistic)": tune(E["own"], True)}
timed("held-out evaluation", t0)

print("\n" + "=" * 86)
print(f"HELD-OUT RESULTS  ({n_ent:,} entities, {int((tc==0).sum()):,} singletons)   "
      f"ceiling: blocked {f05(blk, blk, tc).mean():.4f} | kept {f05(kp, kp, tc).mean():.4f}")
print("=" * 86)
print(f"  {'variant':<44}{'F0.5':>8}{'prec':>8}{'recall':>8}   " + "  ".join(f"{c:>7}" for c in COUNTRIES))
print(f"  {'v2 (same held-out entities, for reference)':<44}{V2_HELDOUT:8.4f}")
print(f"  {'v3 (same held-out entities, for reference)':<44}{V3_HELDOUT:8.4f}")
for name, (th, F, pc, tp) in results.items():
    prec = (tp[pc > 0] / pc[pc > 0]).mean(); rec = (tp[tc > 0] / tc[tc > 0]).mean()
    print(f"  {name:<44}{F.mean():8.4f}{prec:8.4f}{rec:8.4f}   "
          + "  ".join(f"{F[cty==i].mean():7.4f}" for i in range(len(COUNTRIES))))
for i, c in enumerate(COUNTRIES):
    em = cty == i
    print(f"    {c}: blocking recall {blk[em].sum()/max(tc[em].sum(),1)*100:6.2f}%   "
          f"after keep-{N_KEEP[c]} {kp[em].sum()/max(tc[em].sum(),1)*100:6.2f}%")
F_pc = results["v4, per-country thresholds"][1].mean()
F_own = results["v4, per-country + one-owner (pessimistic)"][1].mean()
use_owner = bool(F_own >= F_pc)
best_key = "v4, per-country + one-owner (pessimistic)" if use_owner else "v4, per-country thresholds"
th_final, F_final, pc_f, _ = results[best_key]
has_cand = np.zeros(n_ent, bool); has_cand[np.unique(E["ent"])] = True
rate = {c: float(pc_f[(cty == i) & has_cand].mean()) for i, c in enumerate(COUNTRIES)}
empty = {c: float((pc_f[cty == i] == 0).mean()) for i, c in enumerate(COUNTRIES)}
print(f"\n  thresholds: {', '.join(f'{c} {t:.2f}' for c, t in th_final.items())}   "
      f"(unseen countries: {results['v4, one threshold'][0][COUNTRIES[0]]:.2f})   "
      f"one-owner: {'ON' if use_owner else 'OFF'}   keep: {N_KEEP}")
imp = sorted(zip(matcher.feature_importance("gain"), RICH_V4), reverse=True)
print("  top features: " + ", ".join(n for _, n in imp[:14]))

In [ ]:
# %% [CELL 15] Save
OUT.mkdir(parents=True, exist_ok=True)
prefilter.save_model(str(OUT / "prefilter.txt"))
matcher.save_model(str(OUT / "matcher.txt"), num_iteration=matcher.best_iteration)
config = {"version": 4, "layers": LAYERS, "pre_features": PRE_V3, "rich_features": RICH_V4,
          "n_keep": N_KEEP, "n_keep_unseen": int(max(N_KEEP.values())),
          "thresholds": th_final, "threshold_unseen": results["v4, one threshold"][0][COUNTRIES[0]],
          "one_owner": use_owner, "heldout_f05": float(F_final.mean()),
          "pred_matches_per_entity": rate, "pred_matches_per_entity_all": float(np.mean(list(rate.values()))),
          "pred_empty_rate": empty, "best_iteration": matcher.best_iteration, "smoke": SMOKE,
          "results": {k: float(v[1].mean()) for k, v in results.items()},
          "importance": {n: float(g) for g, n in imp}}
with open(OUT / "config.json", "w") as f:
    json.dump(config, f, indent=2)
with open(OUT / "te_stats.json", "w") as f:                 # learned word rates, used by Step 2 v4
    json.dump(TE["all"], f)
print(f"  v4 vs v3 on the same held-out entities: {F_final.mean():.4f} vs {V3_HELDOUT:.4f} "
      f"({(F_final.mean() - V3_HELDOUT)*100:+.2f} pts)")
print("\n" + "=" * 70)
print(f"STEP 1 v4 {'SMOKE' if SMOKE else 'FULL'} RUN COMPLETE in {(time.time()-T_START)/60:.1f} min")
print("=" * 70)
for fn in sorted(OUT.iterdir()):
    print(f"  {OUT.name}/{fn.name:<16} {fn.stat().st_size/1e6:6.2f} MB")
if SMOKE:
    SCALE = sum(META[c]["full_nq"] for c in COUNTRIES) / max(sum(META[c]["nq"] for c in COUNTRIES), 1)
    project(SCALE, f"full run = {SCALE:.0f}x the smoke queries; matcher rounds may differ")
    print("\n  Smoke run passed. Set SMOKE = False and run again.")
else:
    shutil.rmtree(WORK / "_unzipped", ignore_errors=True)
    print("  Zip stage2_models_v4 and send it to the test side.")
mem("final")